### Topic 8: Embedding Model Migration

### Why Do We Need Embedding Model Migration?

Suppose we built our Qdrant collection using:

```text
Embedding Model:
paraphrase-multilingual-MiniLM-L12-v2
```

Every document was converted into embeddings using this model.

Later, we decide to use a better model.

```text
Embedding Model:
all-MiniLM-L6-v2
```

This process is called **Embedding Model Migration**.

---

### Why Can't We Simply Change the Model?

Suppose the collection already contains embeddings created by the old model.

```text
Old Model

Document 1
Embedding A

Document 2
Embedding B

Document 3
Embedding C
```

Now the user sends a query.

Instead of using the old model, the application uses the new model.

```text
Query

New Model

Query Embedding X
```

Now Qdrant compares:

- Query embedding from the **new model**
- Document embeddings from the **old model**

Since both models generate embeddings differently, the similarity scores become incorrect.

Both the documents and the query **must always use the same embedding model**.

---

### What Is the Correct Solution?

When the embedding model changes:

- Load the new embedding model.
- Generate new embeddings for every document.
- Create a new Qdrant collection.
- Insert all new embeddings.
- Validate the search results.
- Start using the new collection.
- Delete the old collection after verification.

This process is called **Re-indexing**.

---

### Example

Old collection:

```text
Model:
paraphrase-multilingual-MiniLM-L12-v2

FD Interest Rate
FD Closure
FD Penalty
```

New collection:

```text
Model:
all-MiniLM-L6-v2

FD Interest Rate
FD Closure
FD Penalty
```

The document text is the same.

Only the embeddings are regenerated using the new model.

---

### Why Create a New Collection?

Creating a new collection keeps the old collection available while the new one is being built.

After validation, the application starts using the new collection.

If any problem occurs, the application can immediately switch back to the old collection.

---

### Alias Swap

Applications always search using an alias such as:

```text
fd_live
```

Initially:

```text
fd_live
Old Collection
```

After migration:

```text
fd_live
New Collection
```

Only the alias changes.

The application code does not change.

---

### Manifest Problem

Suppose the manifest stores:

```text
FD_Policy.pdf

Hash = abc123
```

The PDF has not changed.

Only the embedding model changed.

The manifest checks only the file hash.

It incorrectly reports:

```text
Nothing to ingest
```

However, all embeddings are now outdated.

The solution is to store both:

- File hash
- Embedding model name

If either changes, regenerate all embeddings.

---

### Real-World Issues, Edge Cases & Debugging

**Changed model without re-indexing**

The application starts using the new model while Qdrant still contains old embeddings.

Result: Search quality becomes poor.

---

**Different vector dimensions**

Old model:

```text
384 dimensions
```

New model:

```text
768 dimensions
```

Result: Qdrant throws a vector dimension mismatch error.

---

**Partial migration**

Some documents use the old model while others use the new model.

Result: Search results become inconsistent.

Always migrate the entire collection.

---

**Manifest does not track model**

The manifest detects only file changes.

Result: Model changes are ignored.

Store the embedding model name in the manifest.

---

### Common Mistakes

- Changing the embedding model without rebuilding the collection.
- Mixing embeddings from different models.
- Updating the existing collection instead of creating a new one.
- Forgetting to validate the new collection.
- Not tracking the embedding model in the manifest.

---

### Lead-Level Interview Questions

**Q1. What is Embedding Model Migration?**

Changing the embedding model used to generate document embeddings and rebuilding the vector collection using the new embeddings.

---

**Q2. Why can't we mix embeddings from two different models?**

Each model generates embeddings differently. Query and document embeddings must be generated using the same model for similarity search to work correctly.

---

**Q3. Why do we create a new collection instead of updating the old one?**

The old collection continues serving requests while the new collection is built and validated. This allows easy rollback if required.

---

**Q4. What is re-indexing?**

Re-indexing means generating new embeddings for every document using the new model and storing them in a new Qdrant collection.

---

**Q5. What is an alias swap?**

The application always searches using an alias such as `fd_live`. During migration, only the alias is updated to point to the new collection. No application code changes are required.

---

**Q6. Why should the manifest track the embedding model?**

A file may remain unchanged while the embedding model changes. Tracking the model ensures that all documents are re-embedded when required.

---

### Key Takeaways

- Query and document embeddings must use the same embedding model.
- Changing the embedding model requires re-indexing.
- Always build a new collection instead of modifying the old one.
- Use aliases for zero-downtime migration.
- Track the embedding model in the manifest.

---

### Code Flow

The program executes in the following order:

1. Load the old and new embedding models.
2. Create a Qdrant client.
3. Build a collection using the old embedding model.
4. Embed all documents with the old model and store them in Qdrant.
5. Search the old collection using the old model. This produces correct results.
6. Search the old collection using the new model without rebuilding the collection. This demonstrates the migration bug.
7. Build a new collection using the new embedding model.
8. Re-embed every document using the new model.
9. Store the new embeddings in the new collection.
10. Validate that search now returns correct results.
11. Simulate an alias swap so the application starts using the new collection.
12. Demonstrate why the manifest cannot detect an embedding model change.
13. Print the migration summary.

---

### Output Explanation

#### Loading Models

```text
Old model: paraphrase-multilingual-MiniLM-L12-v2 (384-dim)
New model: all-MiniLM-L6-v2 (384-dim)
```

This confirms that both embedding models are loaded successfully.

Although both models generate **384-dimensional vectors**, they produce different embeddings.

---

#### Building the Old Collection

```text
Built 'fd_old_model' with 5 points
```

Five document chunks are embedded using the old model and stored in Qdrant.

---

#### Search Using the Old Model

```text
score=0.8909
score=0.5689
score=0.5678
```

The query is embedded using the same model that created the document embeddings.

Since both use the same embedding model, similarity search works correctly and the most relevant FD documents are returned.

---

#### Migration Bug

```text
WRONG: new model query vs old collection
```

The query is embedded using the new model while the collection still contains embeddings from the old model.

No exception is thrown.

Qdrant still returns similarity scores.

However, these scores are no longer reliable because the query and document embeddings were generated by different models.

This is called the **silent migration failure**.

---

#### Building the New Collection

```text
Built 'fd_new_model' with 5 points
```

Every document is embedded again using the new model and stored in a new collection.

The document text remains unchanged.

Only the embeddings change.

---

#### Validation

```text
VALIDATE: new model query vs new collection
```

Now both the query and document embeddings use the new model.

Similarity search works correctly again.

This confirms that the migration was successful.

---

#### Alias Swap

```text
Before swap:
fd_live -> fd_old_model

After swap:
fd_live -> fd_new_model
```

Only the alias changes.

The application continues searching using the alias.

No application code needs to change.

If a problem occurs, the alias can immediately point back to the old collection.

---

#### Manifest Blind Spot

```text
content unchanged
hash unchanged
nothing to ingest
```

The manifest checks only the file hash.

Since the document content has not changed, it incorrectly assumes nothing needs to be processed.

However, the embeddings are outdated because the embedding model changed.

The solution is to store the embedding model name along with the file hash.

---

#### Summary

```text
Old collection : still available
New collection : now live
Rollback : one alias update
```

This confirms the recommended production migration strategy:

- Build a new collection.
- Validate search.
- Update the alias.
- Keep the old collection until migration is verified.

---

In [1]:
"""
qdrant_model_migration.py
---------------------------
Demonstrates what actually happens during an embedding model migration,
and what the correct migration procedure looks like.

Shows:
  1. Build a collection with OLD model -- search works correctly
  2. Search with NEW model against OLD vectors -- silent wrong results
  3. Correct fix: re-embed everything with NEW model into NEW collection
  4. Build-then-swap with collection aliases -- zero downtime pattern
  5. Manifest blind spot -- model name not tracked, migration goes undetected

Uses :memory: mode -- no Docker required.
Install: pip install qdrant-client sentence-transformers
"""

import hashlib
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance,
    VectorParams,
    PointStruct,
)
from sentence_transformers import SentenceTransformer

# old model: 384 dimensions -- what this project currently uses
OLD_MODEL_NAME = "paraphrase-multilingual-MiniLM-L12-v2"
OLD_VECTOR_SIZE = 384

# new model: 384 dimensions too -- in a real migration this might differ
# using all-MiniLM-L6-v2 as the "new" model for demonstration
NEW_MODEL_NAME = "all-MiniLM-L6-v2"
NEW_VECTOR_SIZE = 384

OLD_COLLECTION = "fd_old_model"
NEW_COLLECTION = "fd_new_model"
ALIAS_NAME = "fd_live"  # the alias the application always queries against

client = QdrantClient(":memory:")

CHUNKS = [
    {"text": "Premature withdrawal incurs a 1 percent penalty on the applicable rate.",
     "source": "FD_Policy.json", "doc_type": "policy"},
    {"text": "This penalty does not apply if the FD is closed due to death of the depositor.",
     "source": "FD_Policy.json", "doc_type": "policy"},
    {"text": "Senior citizens receive an additional 0.5 percent interest on all tenures.",
     "source": "FD_Product_Guide.json", "doc_type": "product"},
    {"text": "What is the penalty for early FD closure? A 1 percent penalty applies.",
     "source": "FD_FAQ.json", "doc_type": "faq"},
    {"text": "The FD interest rate for 24 months is 7.1 percent per annum.",
     "source": "FD_Product_Guide.json", "doc_type": "product"},
]


def make_point_id(source: str, index: int) -> int:
    # deterministic ID -- same chunk always gets the same ID
    return int(hashlib.md5(f"{source}::{index}".encode()).hexdigest()[:8], 16)


def build_collection(collection_name: str, model: SentenceTransformer,
                     vector_size: int, chunks: list) -> None:
    """Creates a collection and upserts all chunks embedded with the given model."""

    # create the collection with the specified vector size
    client.create_collection(
        collection_name=collection_name,
        vectors_config=VectorParams(size=vector_size, distance=Distance.COSINE),
    )

    # embed all chunks in one batch using the provided model
    texts = [c["text"] for c in chunks]
    embeddings = model.encode(texts, normalize_embeddings=True)

    # build points with deterministic IDs
    points = [
        PointStruct(
            id=make_point_id(chunks[i]["source"], i),
            vector=embeddings[i].tolist(),
            payload={
                "text": chunks[i]["text"],
                "source": chunks[i]["source"],
                "doc_type": chunks[i]["doc_type"],
                # store which model produced these vectors -- key for debugging later
                "embed_model": model._modules['0'].auto_model.config._name_or_path
                if hasattr(model._modules.get('0', None), 'auto_model') else "unknown",
            },
        )
        for i in range(len(chunks))
    ]

    client.upsert(collection_name=collection_name, points=points)
    print(f"Built '{collection_name}' with {len(points)} points "
          f"(vector_size={vector_size})")


def search(collection_name: str, query: str, query_model: SentenceTransformer,
           k: int = 3, label: str = "") -> None:
    """Search a collection using a specific model to embed the query."""

    # embed the query with the provided model
    query_vector = query_model.encode(query, normalize_embeddings=True).tolist()

    results = client.query_points(
        collection_name=collection_name,
        query=query_vector,
        limit=k,
        with_payload=True,
    ).points

    print(f"\n{label}")
    print(f"  Query model : {query_model._first_module().auto_model.config._name_or_path
          if hasattr(query_model._first_module(), 'auto_model') else 'unknown'}")
    print(f"  Collection  : {collection_name}")
    for r in results:
        print(f"  score={r.score:.4f} | {r.payload['text'][:65]}")


def demonstrate_manifest_blind_spot(manifest: dict, current_model: str) -> None:
    """Shows that the manifest as built doesn't detect a model change.
    The fix: store embed_model_name in each manifest entry."""

    print("\n--- Manifest blind spot ---")

    # simulate what the manifest currently tracks (content hash only)
    print("  Current manifest entry (content hash only):")
    print(f"    {{'hash': 'abc123...', 'last_ingested': '2024-01-01'}}")
    print("  After model swap -- manifest still says nothing to ingest:")
    print(f"    content unchanged -> hash unchanged -> 'nothing to ingest'")
    print(f"    but vectors are now stale relative to new model!")

    print("\n  Fixed manifest entry (tracks model name too):")
    print(f"    {{'hash': 'abc123...', 'embed_model': '{current_model}', "
          f"'last_ingested': '2024-01-01'}}")
    print(f"  After model swap to 'all-MiniLM-L6-v2':")
    print(f"    embed_model mismatch detected -> triggers full re-embed")
    print(f"    even though source content (and hash) is unchanged")


# ======================================================================
# Step 1: load both models
# ======================================================================
print("Loading models (may download on first run)...")
old_model = SentenceTransformer(OLD_MODEL_NAME)
new_model = SentenceTransformer(NEW_MODEL_NAME)
print(f"Old model: {OLD_MODEL_NAME} ({OLD_VECTOR_SIZE}-dim)")
print(f"New model: {NEW_MODEL_NAME} ({NEW_VECTOR_SIZE}-dim)")

# ======================================================================
# Step 2: build old collection -- the starting state before migration
# ======================================================================
print("\n--- Step 1: Build collection with OLD model ---")
build_collection(OLD_COLLECTION, old_model, OLD_VECTOR_SIZE, CHUNKS)

# search with old model against old collection -- correct, this works
QUERY = "What is the penalty for closing an FD early?"
search(OLD_COLLECTION, QUERY, old_model,
       label="CORRECT: old model query vs old collection")

# ======================================================================
# Step 3: demonstrate the failure -- search with new model against old
# collection. No error, but scores are meaningless.
# ======================================================================
print("\n--- Step 2: Swap model in code but forget to re-index (THE BUG) ---")
search(OLD_COLLECTION, QUERY, new_model,
       label="WRONG: new model query vs old collection (silent failure)")
print("  No error thrown. Scores look plausible but measure nothing real.")
print("  This is the failure mode -- silent, no exception, wrong answers.")

# ======================================================================
# Step 4: correct migration -- build new collection, validate, swap alias
# ======================================================================
print("\n--- Step 3: Correct migration -- build new collection first ---")
build_collection(NEW_COLLECTION, new_model, NEW_VECTOR_SIZE, CHUNKS)

# validate new collection with new model before swapping
search(NEW_COLLECTION, QUERY, new_model,
       label="VALIDATE: new model query vs new collection (correct)")

# ======================================================================
# Step 5: alias swap -- atomic pointer change, zero downtime
# In-memory mode doesn't support aliases, so we simulate the pattern.
# With Docker/Qdrant Cloud, this would be:
#   client.update_collection_aliases(change_aliases_operations=[
#       CreateAliasOperation(create_alias=CreateAlias(
#           collection_name=NEW_COLLECTION, alias_name=ALIAS_NAME))
#   ])
# ======================================================================
print("\n--- Step 4: Alias swap (simulated -- requires Docker for real aliases) ---")
print(f"  Before swap: alias '{ALIAS_NAME}' -> '{OLD_COLLECTION}'")
print(f"  After swap:  alias '{ALIAS_NAME}' -> '{NEW_COLLECTION}'")
print(f"  This is one API call. Live queries see the new collection instantly.")
print(f"  Old collection stays intact until new one is confirmed working.")
print(f"  Rollback = one API call to point alias back.")

# ======================================================================
# Step 6: manifest blind spot
# ======================================================================
manifest = {
    "FD_Policy.json": {"hash": "abc123", "last_ingested": "2024-01-01"},
}
demonstrate_manifest_blind_spot(manifest, OLD_MODEL_NAME)

# ======================================================================
# Summary
# ======================================================================
print("\n--- Summary ---")
print("  Old collection (old model)  : still exists, serves old traffic via alias")
print("  New collection (new model)  : built, validated, now live via alias")
print("  What changed                : alias pointer (one API call)")
print("  What stayed the same        : all application query code")
print("  Rollback                    : one alias update, instant")
print("  Manifest fix needed         : track embed_model alongside content hash")


Loading models (may download on first run)...
Old model: paraphrase-multilingual-MiniLM-L12-v2 (384-dim)
New model: all-MiniLM-L6-v2 (384-dim)

--- Step 1: Build collection with OLD model ---
Built 'fd_old_model' with 5 points (vector_size=384)

CORRECT: old model query vs old collection
  Query model : sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
  Collection  : fd_old_model
  score=0.8909 | What is the penalty for early FD closure? A 1 percent penalty app
  score=0.5689 | Premature withdrawal incurs a 1 percent penalty on the applicable
  score=0.5678 | This penalty does not apply if the FD is closed due to death of t

--- Step 2: Swap model in code but forget to re-index (THE BUG) ---

WRONG: new model query vs old collection (silent failure)
  Query model : sentence-transformers/all-MiniLM-L6-v2
  Collection  : fd_old_model
  score=0.3980 | What is the penalty for early FD closure? A 1 percent penalty app
  score=0.3188 | This penalty does not apply if the FD is clo